[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Relatividad_%28REL%29/REL-07_Relatividad_Ondas_Gravitacionales.ipynb)


# Investigación Avanzada: Relatividad Ondas Gravitacionales

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def generate_gravitational_wave(m1_msun, m2_msun, distance_mpc, f_min=20.0):
    """
    Generate Post-Newtonian (leading order) gravitational waveform for an inspiraling binary.
    m1_msun, m2_msun: masses in solar masses
    distance_mpc: luminosity distance in Mpc
    f_min: starting frequency in Hz
    """
    # Constants in SI units
    G = 6.67430e-11
    c = 2.99792458e8
    M_sun = 1.98847e30
    Mpc = 3.08567758149137e22
    
    m1 = m1_msun * M_sun
    m2 = m2_msun * M_sun
    M_tot = m1 + m2
    mu = m1 * m2 / M_tot
    M_chirp = mu**(3.0/5.0) * M_tot**(2.0/5.0)
    D = distance_mpc * Mpc
    
    # Characteristic time to coalescence from f_min
    # t_c - t = 5/256 * c^5 / (G^3 M_chirp^5) * (pi f)^(-8/3)
    t_c = 5.0/256.0 * c**5 / (G**(5.0/3.0) * M_chirp**(5.0/3.0)) * (np.pi * f_min)**(-8.0/3.0)
    
    # Generate time array up to ISCO (Innermost Stable Circular Orbit)
    # f_ISCO = c^3 / (6^(3/2) pi G M_tot)
    f_isco = c**3 / (6.0**(1.5) * np.pi * G * M_tot)
    t_isco = t_c - 5.0/256.0 * c**5 / (G**(5.0/3.0) * M_chirp**(5.0/3.0)) * (np.pi * f_isco)**(-8.0/3.0)
    
    # We sample at 4096 Hz
    fs = 4096
    t = np.arange(0, t_isco, 1.0/fs)
    
    # Time to coalescence array
    tau = t_c - t
    
    # Frequency evolution
    f = (1.0 / np.pi) * (5.0 / 256.0 * c**5 / (G**(5.0/3.0) * M_chirp**(5.0/3.0)) * (1.0 / tau))**(3.0/8.0)
    
    # Phase evolution
    # Phi(tau) = -2 (5G M_chirp / c^3)^(-5/8) tau^(5/8) + Phi_c
    Phi = -2.0 * (5.0 * G * M_chirp / c**3)**(-5.0/8.0) * tau**(5.0/8.0)
    
    # Amplitude evolution
    # A = 4/D * (G M_chirp / c^2)^(5/3) * (pi f / c)^(2/3)
    A = 4.0 / D * (G * M_chirp / c**2)**(5.0/3.0) * (np.pi * f / c)**(2.0/3.0)
    
    # Plus and cross polarizations (assuming optimal orientation i=0)
    h_plus = A * np.cos(Phi)
    h_cross = A * np.sin(Phi)
    
    return t, h_plus, h_cross, f

if __name__ == '__main__':
    # Binary Black Hole similar to GW150914
    m1 = 36.0
    m2 = 29.0
    dist = 410.0 # Mpc
    
    t, hp, hc, f = generate_gravitational_wave(m1, m2, dist, f_min=30)
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(2, 1, 1)
    plt.plot(t, hp, label=f'$h_+$ (m1={m1}$M_\odot$, m2={m2}$M_\odot$)', color='blue')
    plt.ylabel('Strain')
    plt.title('Gravitational Wave Strain (Leading Order PN)')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 1, 2)
    plt.plot(t, f, color='red')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (s)')
    plt.yscale('log')
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('Gravitational_Waves_Inspiral.png', dpi=300)
    print("GW simulation complete. Image saved.")
